# Runtime clássico V2 — setup + sliders + webcam

Fluxo de uso:

1. Execute a célula **Selecionar setup paramétrico** e escolha o setup na lista suspensa.
2. Execute a célula **Ajustar parâmetros do runtime** e ajuste os sliders.
3. Execute a célula **Abrir webcam**.

Os sliders são widgets do notebook, não são os sliders OpenCV/Qt. Isso evita o problema da janela de controles que impediu a webcam de abrir.

Este notebook deve ficar em `vision/cylinders_detect/classic_vision/`, junto com `_runtime_classico_v2_core.py`.


In [ ]:
# Selecionar setup paramétrico
from pathlib import Path
import sys
from IPython.display import display, clear_output
import ipywidgets as widgets

# Garante que o notebook consiga importar o arquivo auxiliar do runtime V2.
THIS_DIR = Path.cwd()
if not (THIS_DIR / "_runtime_classico_v2_core.py").exists():
    candidatos = [
        Path.cwd() / "vision" / "cylinders_detect" / "classic_vision",
        ]
    for c in candidatos:
        if (c / "_runtime_classico_v2_core.py").exists():
            THIS_DIR = c
            break

if str(THIS_DIR) not in sys.path:
    sys.path.insert(0, str(THIS_DIR))

import _runtime_classico_v2_core as rt

setups = rt.carregar_todos_setups()
params_meta, meta = rt.carregar_params_metadata()

opcoes = []
for key, s in setups.items():
    label = f"{key} | {s.get('nome', '')} | origem={s.get('origem', '-') }"
    opcoes.append((label, ("setup", key)))

if params_meta is not None:
    label = f"metadata | parâmetros do último treinamento | setup={meta.get('setup_id', '-')} | nome={meta.get('setup_nome', '-')}"
    opcoes.append((label, ("metadata", "metadata")))

if not opcoes:
    raise RuntimeError("Nenhum setup JSON ou metadata de treinamento foi encontrado.")

valor_padrao = next(
    (
        valor
        for _, valor in opcoes
        if valor == ("metadata", "metadata")
    ),
    opcoes[0][1],
)

setup_dropdown = widgets.Dropdown(
    options=opcoes,
    value=valor_padrao,
    description="Setup:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "80px"},
)

saida_setup = widgets.Output()
PARAMS_SELECIONADOS = None
SETUP_DESC = None

def aplicar_setup(change=None):
    global PARAMS_SELECIONADOS, SETUP_DESC
    tipo, valor = setup_dropdown.value
    if tipo == "metadata":
        PARAMS_SELECIONADOS, SETUP_DESC = rt.escolher_params("metadata")
    else:
        PARAMS_SELECIONADOS, SETUP_DESC = rt.escolher_params(valor)
    with saida_setup:
        clear_output(wait=True)
        print(f"Setup selecionado: {SETUP_DESC}")
        print(f"Total de parâmetros carregados: {len(PARAMS_SELECIONADOS)}")
        print("Agora execute a célula: Ajustar parâmetros do runtime.")

setup_dropdown.observe(aplicar_setup, names="value")
aplicar_setup()
display(setup_dropdown, saida_setup)


In [ ]:
# Ajustar parâmetros do runtime
import ipywidgets as widgets
from IPython.display import display, clear_output

camera_widget = widgets.IntText(value=0, description="Câmera:", layout=widgets.Layout(width="220px"))
width_widget = widgets.Dropdown(options=[("320", 320), ("480", 480), ("640", 640), ("800", 800), ("1280", 1280)], value=320, description="Largura:", layout=widgets.Layout(width="220px"))
height_widget = widgets.Dropdown(options=[("240", 240), ("360", 360), ("480", 480), ("600", 600), ("720", 720)], value=240, description="Altura:", layout=widgets.Layout(width="220px"))

detect_every_widget = widgets.IntSlider(value=10, min=1, max=30, step=1, description="Detect. a cada:", continuous_update=False, layout=widgets.Layout(width="360px"), style={"description_width": "120px"})
max_rois_widget = widgets.IntSlider(value=80, min=5, max=200, step=5, description="Máx. ROIs:", continuous_update=False, layout=widgets.Layout(width="360px"), style={"description_width": "120px"})
score_min_widget = widgets.FloatSlider(value=-5.0, min=-20.0, max=5.0, step=0.5, description="Score mín:", continuous_update=False, readout_format=".1f", layout=widgets.Layout(width="360px"), style={"description_width": "120px"})
nms_iou_widget = widgets.FloatSlider(value=0.30, min=0.05, max=0.90, step=0.05, description="NMS IoU:", continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="360px"), style={"description_width": "120px"})
max_det_widget = widgets.IntSlider(value=20, min=1, max=50, step=1, description="Máx. detecções:", continuous_update=False, layout=widgets.Layout(width="360px"), style={"description_width": "120px"})
display_scale_widget = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description="Escala janela:", continuous_update=False, readout_format=".1f", layout=widgets.Layout(width="360px"), style={"description_width": "120px"})

mirror_widget = widgets.Checkbox(value=False, description="Espelhar webcam")
async_widget = widgets.Checkbox(value=True, description="Detecção assíncrona")

saida_runtime = widgets.Output()

def atualizar_resumo(change=None):
    global CAMERA_INDEX, WIDTH, HEIGHT, DETECT_EVERY, MAX_ROIS_RUNTIME, SCORE_MIN, NMS_IOU, MAX_DET, MIRROR, DISPLAY_SCALE, ASYNC_DETECTION
    CAMERA_INDEX = int(camera_widget.value)
    WIDTH = int(width_widget.value)
    HEIGHT = int(height_widget.value)
    DETECT_EVERY = int(detect_every_widget.value)
    MAX_ROIS_RUNTIME = int(max_rois_widget.value)
    SCORE_MIN = float(score_min_widget.value)
    NMS_IOU = float(nms_iou_widget.value)
    MAX_DET = int(max_det_widget.value)
    MIRROR = bool(mirror_widget.value)
    DISPLAY_SCALE = float(display_scale_widget.value)
    ASYNC_DETECTION = bool(async_widget.value)
    with saida_runtime:
        clear_output(wait=True)
        print("Parâmetros prontos para abrir a webcam:")
        print(f"camera={CAMERA_INDEX} | resolução={WIDTH}x{HEIGHT} | detect_every={DETECT_EVERY} | max_rois={MAX_ROIS_RUNTIME}")
        print(f"score_min={SCORE_MIN:.1f} | nms_iou={NMS_IOU:.2f} | max_det={MAX_DET} | mirror={MIRROR} | async={ASYNC_DETECTION}")
        print("Depois execute a célula: Abrir webcam com os sliders atuais.")

for w in [camera_widget, width_widget, height_widget, detect_every_widget, max_rois_widget, score_min_widget, nms_iou_widget, max_det_widget, display_scale_widget, mirror_widget, async_widget]:
    w.observe(atualizar_resumo, names="value")

linha1 = widgets.HBox([camera_widget, width_widget, height_widget])
linha2 = widgets.HBox([detect_every_widget, max_rois_widget])
linha3 = widgets.HBox([score_min_widget, nms_iou_widget])
linha4 = widgets.HBox([max_det_widget, display_scale_widget])
linha5 = widgets.HBox([mirror_widget, async_widget])

atualizar_resumo()
display(linha1, linha2, linha3, linha4, linha5, saida_runtime)


In [ ]:
# Abrir webcam com o setup selecionado e os sliders atuais
if "PARAMS_SELECIONADOS" not in globals() or PARAMS_SELECIONADOS is None:
    raise RuntimeError("Execute primeiro a célula de seleção de setup.")

# Garante valores padrão caso a célula de sliders ainda não tenha sido executada.
CAMERA_INDEX = globals().get("CAMERA_INDEX", 0)
WIDTH = globals().get("WIDTH", 320)
HEIGHT = globals().get("HEIGHT", 240)
DETECT_EVERY = globals().get("DETECT_EVERY", 10)
MAX_ROIS_RUNTIME = globals().get("MAX_ROIS_RUNTIME", 80)
SCORE_MIN = globals().get("SCORE_MIN", -5.0)
NMS_IOU = globals().get("NMS_IOU", 0.30)
MAX_DET = globals().get("MAX_DET", 20)
MIRROR = globals().get("MIRROR", False)
DISPLAY_SCALE = globals().get("DISPLAY_SCALE", 1.0)
ASYNC_DETECTION = globals().get("ASYNC_DETECTION", True)

print("Setup usado:", SETUP_DESC)
print("Carregando modelo:", rt.MODEL_PATH)
print(f"Runtime: camera={CAMERA_INDEX}, resolução={WIDTH}x{HEIGHT}, detect_every={DETECT_EVERY}, max_rois={MAX_ROIS_RUNTIME}, score_min={SCORE_MIN}")
clf = rt.carregar_modelo(rt.MODEL_PATH)

rt.rodar_webcam(
    params=PARAMS_SELECIONADOS,
    clf=clf,
    camera_index=CAMERA_INDEX,
    score_min=SCORE_MIN,
    nms_iou=NMS_IOU,
    max_det=MAX_DET,
    width=WIDTH,
    height=HEIGHT,
    espelhar=MIRROR,
    resize_display=DISPLAY_SCALE,
    detect_every=DETECT_EVERY,
    max_rois_runtime=MAX_ROIS_RUNTIME,
    async_detection=ASYNC_DETECTION,
    controles=False,
)
